In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(2*np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)))

def eval_metrics(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "SMAPE": float(smape(y_true, y_pred)),
    }

def split_by_day(df, test_days=7, val_days=7):
    max_day = int(df["d"].max())
    test_start = max_day - test_days + 1
    val_start  = test_start - val_days

    train_df = df[df["d"] < val_start].copy()
    val_df   = df[(df["d"] >= val_start) & (df["d"] < test_start)].copy()
    test_df  = df[df["d"] >= test_start].copy()

    return train_df, val_df, test_df

# 補零
def densify_topk_series(df_raw: pd.DataFrame, top_k=20000):
    """
    df_raw 需要欄位: d,t,x,y,count
    回傳：只含 top_k 個 (x,y,t) 且已補齊所有 d 的 DataFrame
    """
    df = df_raw[["d","t","x","y","count"]].copy()

    # 選最活躍的 (x,y,t)：用總量或出現天數都可以
    key_sum = df.groupby(["x","y","t"])["count"].sum().sort_values(ascending=False)
    top_keys = key_sum.head(top_k).index

    df = df.set_index(["x","y","t"]).loc[top_keys].reset_index()

    dmin, dmax = int(df["d"].min()), int(df["d"].max())
    all_d = np.arange(dmin, dmax + 1, dtype=int)

    out = []
    for (x,y,t), g in df.groupby(["x","y","t"], sort=False):
        g2 = g.set_index("d").reindex(all_d)
        g2["count"] = g2["count"].fillna(0.0)
        g2["d"] = all_d
        g2["x"] = x; g2["y"] = y; g2["t"] = t
        out.append(g2[["d","t","x","y","count"]])

    return pd.concat(out, ignore_index=True)

In [3]:
import holidays

ORIGIN_DATE = pd.Timestamp("2019-09-06")  # d=0

def add_calendar_features(
    df: pd.DataFrame,
    origin_date: str | pd.Timestamp = ORIGIN_DATE,
    country: str = "JP",
    events: list[dict] | None = None,
) -> pd.DataFrame:
    """
    df needs: d (int day index)
    outputs: date, month, season, is_holiday, event_code, is_event, (optional) weekday/is_weekend
    """
    df = df.copy()
    origin_date = pd.Timestamp(origin_date)

    # d -> date
    df["date"] = origin_date + pd.to_timedelta(df["d"].astype(int), unit="D")
    df["month"] = df["date"].dt.month.astype(np.int16)

    # season: 0=Winter(12-2),1=Spring(3-5),2=Summer(6-8),3=Autumn(9-11)
    m = df["month"].to_numpy()
    season = np.zeros(len(df), dtype=np.int16)
    season[(m >= 3) & (m <= 5)] = 1
    season[(m >= 6) & (m <= 8)] = 2
    season[(m >= 9) & (m <= 11)] = 3
    df["season"] = season

    # holiday flag (JP): use python package "holidays" if available; otherwise fallback to False / user list
    is_holiday = np.zeros(len(df), dtype=bool)
    jp_h = holidays.country_holidays(country)
    is_holiday = df["date"].dt.date.map(lambda x: x in jp_h).to_numpy()


    df["is_holiday"] = is_holiday.astype(np.int8)

    # event flags (user-provided ranges)
    # events format:
    # [{"name":"snow_festival", "start":"2023-02-xx", "end":"2023-02-xx"}, ...]
    df["event_code"] = 0
    if events:
        for i, ev in enumerate(events, start=1):
            s = pd.Timestamp(ev["start"])
            e = pd.Timestamp(ev["end"])
            mask = (df["date"] >= s) & (df["date"] <= e)
            df.loc[mask, "event_code"] = i
    df["is_event"] = (df["event_code"] > 0).astype(np.int8)

    # 若你 df 裡 weekday/is_weekend 不可靠，可用 date 重算（Mon=0..Sun=6）
    '''if "weekday" not in df.columns:
        df["weekday"] = df["date"].dt.weekday.astype(np.int16)
    if "is_weekend" not in df.columns:
        df["is_weekend"] = (df["weekday"] >= 5).astype(np.int8)'''

    return df

In [4]:
#freq_nz(x,y,t) = 過去這格在這個時段出現 count>0 的比例
def build_priors(train_df: pd.DataFrame, keys: list[str], alpha: float = 1.0) -> pd.DataFrame:
    """
    returns: keys + [freq_nz, base_log]
    freq_nz: smoothed P(count>0)
    base_log: mean(log1p(count))
    """
    g = train_df.groupby(keys, as_index=False)

    agg = g["count"].agg(
        n="size",
        nz=lambda s: int((s.to_numpy() > 0).sum()),
        base_log=lambda s: float(np.log1p(s.to_numpy()).mean()),
    )

    # smoothed frequency prior
    agg["freq_nz"] = (agg["nz"] + alpha) / (agg["n"] + 2 * alpha)
    return agg[keys + ["freq_nz", "base_log"]]

def attach_priors_with_backoff(df: pd.DataFrame,
                              priA: pd.DataFrame, keysA: list[str],
                              priB: pd.DataFrame, keysB: list[str],
                              priC: pd.DataFrame, keysC: list[str]) -> pd.DataFrame:
    df = df.copy()

    # merge A
    df = df.merge(priA, on=keysA, how="left", suffixes=("", "_A"))
    # merge B
    df = df.merge(priB, on=keysB, how="left", suffixes=("", "_B"))
    # merge C
    df = df.merge(priC, on=keysC, how="left", suffixes=("", "_C"))

    # choose base_log backoff: A -> B -> C -> global(0)
    df["base_log"] = df["base_log"].combine_first(df["base_log_B"]).combine_first(df["base_log_C"]).fillna(0.0).astype(np.float32)

    # choose freq_nz backoff: A -> B -> C -> global(small)
    df["freq_nz"] = df["freq_nz"].combine_first(df["freq_nz_B"]).combine_first(df["freq_nz_C"]).fillna(0.0).astype(np.float32)

    # cleanup helper cols
    drop_cols = [c for c in df.columns if c.endswith("_B") or c.endswith("_C")]
    df.drop(columns=drop_cols, inplace=True)

    return df

In [5]:
df = pd.read_parquet("../data/processed/sapporo_density.parquet")
df = df[~((df["x"]==999) & (df["y"]==999))].copy()

df_raw = df.copy()
df_dense = df_raw.copy()
df_dense.shape
# 只補 top_k，先用小一點，穩了再加
#df_dense = densify_topk_series(df_raw, top_k=20000)
#df_dense = pd.read_parquet("../data/processed/sapporo_density_filled_zero.parquet").copy() #補零後的
#print(df_dense.shape, df_dense["count"].mean(), (df_dense["count"]==0).mean())

(6024088, 5)

In [6]:
# LSTM 序列長度
SEQ_LEN = 7

df = df_dense.copy() 

# d=0 是星期日
df["date"] = pd.to_datetime("2019-09-15") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# 排序
df = df.sort_values(["x","y","t","d"])

# 產生 lag_1..lag_SEQ_LEN（給 LSTM 當序列，也給 baseline）
g = df.groupby(["x","y","t"])["count"]
for k in range(1, SEQ_LEN+1):
    df[f"lag_{k}"] = g.shift(k)

# rolling 特徵
df["rolling_3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["rolling_7"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())

'''# 丟掉 lag 不足的列
need_cols = [f"lag_{k}" for k in range(1, SEQ_LEN+1)] + ["rolling_3","rolling_7"]
df = df.dropna(subset=need_cols).copy()'''
# 只要求 lag_1 不為 NaN（至少有前一天的觀測），其餘允許缺失
df = df.dropna(subset=["lag_1"]).copy()

train_df, val_df, test_df = split_by_day(df, test_days=7, val_days=7)
print(len(train_df), len(val_df), len(test_df))

# calendar features（train/val/test 都要）
events = [
    {"name":"rugbyworldcup", "start":"2019-09-20", "end":"2019-11-02"},
]
train_df = add_calendar_features(train_df, events=events)
val_df   = add_calendar_features(val_df,   events=events)
test_df  = add_calendar_features(test_df,  events=events)

# priors keys
keysA = ["x","y","t","weekday","is_holiday","event_code","month"]  # 或把 month 換成 season
keysB = ["x","y","t","weekday"]
keysC = ["x","y","t"]

priA = build_priors(train_df, keysA, alpha=1.0)
priB = build_priors(train_df, keysB, alpha=1.0)
priC = build_priors(train_df, keysC, alpha=1.0)

train_df = attach_priors_with_backoff(train_df, priA, keysA, priB, keysB, priC, keysC)
val_df   = attach_priors_with_backoff(val_df,   priA, keysA, priB, keysB, priC, keysC)
test_df  = attach_priors_with_backoff(test_df,  priA, keysA, priB, keysB, priC, keysC)

# residual target（ConvLSTM.py 的 Dataset 已支援 y_target / base_log，不改架構）
train_df["y_target"] = (np.log1p(train_df["count"].to_numpy(np.float32)) - train_df["base_log"].to_numpy(np.float32)).astype(np.float32)
val_df["y_target"]   = (np.log1p(val_df["count"].to_numpy(np.float32))   - val_df["base_log"].to_numpy(np.float32)).astype(np.float32)
test_df["y_target"]  = (np.log1p(test_df["count"].to_numpy(np.float32))  - test_df["base_log"].to_numpy(np.float32)).astype(np.float32)

4758134 527508 523272


In [7]:
def baseline_lagk(df_split, k=7):
    return df_split[f"lag_{k}"].to_numpy()

def fit_baseline_hist(train_df):
    # (weekday,t,x,y) 的歷史平均
    hist = train_df.groupby(["weekday","t","x","y"])["count"].mean()
    return hist

def predict_baseline_hist(df_split, hist_series):
    key = list(zip(df_split["weekday"], df_split["t"], df_split["x"], df_split["y"]))
    # 沒看過的 key 用全域平均補
    global_mean = float(hist_series.mean())
    pred = np.array([hist_series.get(k, global_mean) for k in key], dtype=float)
    return pred

In [8]:
import torch, pickle

def save_lstm_pkl(model, path_pkl, config: dict):
    payload = {
        "state_dict": model.state_dict(),
        "config": config
    }
    torch.save(payload, path_pkl)   # 用 torch 的序列化最穩（副檔名可叫 .pkl）
    # 或你也可以用 pickle.dump(payload, open(...,"wb"))，但 torch.save 更常用

config = {
    "seq_len": SEQ_LEN,
    "hidden": 64,
    "layers": 1,
    "lr": 1e-3,

    "n_weekday": 7,
    "n_t": 48,
    "n_x": int(df["x"].max()) + 1,
    "n_y": int(df["y"].max()) + 1,

    "emb_wd": 2,
    "emb_t": 8,
    "emb_x": 16,
    "emb_y": 16,
}



In [ ]:
from src.LSTM import *
from src.ConvLSTM import *

def run_all_models(train_df, val_df, test_df, seq_len=14):
    results = []

    # ---- baselines ----
    hist = fit_baseline_hist(train_df)

    for _df in [train_df, val_df, test_df]:
        _df["y_hist"] = predict_baseline_hist(_df, hist)              # baseline in count scale
        _df["base_log"] = np.log1p(_df["y_hist"].to_numpy())          # log1p baseline
        _df["y_target"] = np.log1p(_df["count"].to_numpy()) - _df["base_log"]  # residual in log space


    # 檢查殘差大小
    print("y_target std (越小越好):", train_df["y_target"].std())
    print("log1p(count) std:", np.log1p(train_df["count"]).std())


    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()

        yhat = predict_baseline_hist(df_split, hist)
        results.append(eval_metrics(f"baseline_hist ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=1)
        results.append(eval_metrics(f"baseline_lag1 ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=7)
        results.append(eval_metrics(f"baseline_lag7 ({split_name})", y, yhat))

    # ---- ConvLSTM ----
    # ConvLSTM is heavier; use smaller sample_n/batch_size first.
    conv_spec = {
        "hid_ch": 64,
        "kernel_size": 3,
        "patch_radius": 4,   
        "mlp": 256,

        "n_weekday": 7,
        "n_t": 48,
        "n_x": int(pd.concat([train_df["x"], val_df["x"]]).max()) + 1,
        "n_y": int(pd.concat([train_df["y"], val_df["y"]]).max()) + 1,
        "emb_wd": 4,
        "emb_t": 16,
        "emb_x": 32,
        "emb_y": 32,

        "seq_len": SEQ_LEN,
        "epochs": 40,
        "lr": 1e-3,
        "batch_size": 512,
        "sample_n": 500_000,
        "seed": 42,
    }

    conv_model = ConvLSTMRegEmbed(
        hid_ch=conv_spec["hid_ch"],
        kernel_size=conv_spec["kernel_size"],
        n_weekday=conv_spec["n_weekday"],
        n_t=conv_spec["n_t"],
        n_x=conv_spec["n_x"],
        n_y=conv_spec["n_y"],
        emb_wd=conv_spec["emb_wd"],
        emb_t=conv_spec["emb_t"],
        emb_x=conv_spec["emb_x"],
        emb_y=conv_spec["emb_y"],
        mlp=conv_spec["mlp"],
    )

    # Build lookup on (train+val+test) so val/test lags exist (same idea as your lag features)
    lookup_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

    convlstm = train_convlstm_embed(
        model=conv_model,
        train_df=train_df,
        val_df=val_df,
        seq_len=conv_spec["seq_len"],
        patch_radius=conv_spec["patch_radius"],
        sample_n=conv_spec["sample_n"],
        batch_size=conv_spec["batch_size"],
        epochs=conv_spec["epochs"],
        lr=conv_spec["lr"],
        seed=conv_spec["seed"],
        loss="huber",
        huber_beta=1.0,
        use_residual=True,
        lookup_df=lookup_df,
        neg_ratio=0.0,      
        w_nonzero=1.0, 
        freq_prior_df=priB
    )

    # optional: save model
    try:
        save_convlstm_pkl(convlstm, "../models/convlstm_sapporo.pkl", {
            **config,
            "hid_ch": conv_spec["hid_ch"],
            "kernel_size": conv_spec["kernel_size"],
            "patch_radius": conv_spec["patch_radius"],
            "mlp": conv_spec["mlp"],
        })
    except Exception as e:
        print("save_convlstm_pkl skipped:", e)

    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()
        yhat = predict_convlstm_embed(
            convlstm,
            df_split,
            seq_len=seq_len,
            patch_radius=conv_spec["patch_radius"],
            batch_size=512,
            use_residual=True,
            lookup_df=lookup_df,
            freq_prior_df=priB
        )
        results.append(eval_metrics(f"ConvLSTM ({split_name})", y, yhat))

    y = test_df["count"].to_numpy()
    mask_nz = y > 0

    print("=== 基本分布 ===")
    print(f"非零樣本數: {mask_nz.sum()} | 零值樣本數: {(~mask_nz).sum()}")
    print(f"count 中位數: {np.median(y[mask_nz]):.1f}")
    print(f"count 90th:   {np.percentile(y[mask_nz], 90):.1f}")
    print(f"count 99th:   {np.percentile(y[mask_nz], 99):.1f}")
    print(f"count max:    {y[mask_nz].max():.1f}")

    print("\n=== 預測品質分層 ===")
    print(f"整體   R²: {r2_score(y, yhat):.4f}")
    print(f"非零   R²: {r2_score(y[mask_nz], yhat[mask_nz]):.4f}")

    # 按 count 大小分層看 R²
    for lo, hi in [(1,5),(5,20),(20,100),(100,99999)]:
        m = (y >= lo) & (y < hi)
        if m.sum() > 10:
            print(f"  count [{lo:4d},{hi:5d}): n={m.sum():6d}  R²={r2_score(y[m],yhat[m]):.4f}")

    print("\n=== 殘差分析 ===")
    residual = yhat[mask_nz] - y[mask_nz]
    print(f"殘差 bias (mean): {residual.mean():.4f}")
    print(f"殘差 std:         {residual.std():.4f}")

    print("\n=== base_log 品質 ===")
    base_pred = np.expm1(test_df["base_log"].to_numpy())
    print(f"base_log R²: {r2_score(y, base_pred):.4f}  ← baseline 本身有多強")

    print("\n=== 預測值分布 ===")
    print(f"yhat > 0.5 的比例: {(yhat > 0.5).mean():.1%}  (實際: {mask_nz.mean():.1%})")
    print(f"yhat 中位數(非零預測): {np.median(yhat[yhat>0.5]):.2f}")

    return pd.DataFrame(results).sort_values(["model"]).reset_index(drop=True)

report = run_all_models(train_df, val_df, test_df, seq_len=SEQ_LEN)
report